<a href="https://colab.research.google.com/github/jensenjenschristian/ragprojekt/blob/main/week2_embedding_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
if os.path.exists("/content/ragprojekt"):
    !git -C /content/ragprojekt pull
else:
    !git clone https://github.com/jensenjenschristian/ragprojekt.git /content/ragprojekt

Cloning into '/content/ragprojekt'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 111 (delta 50), reused 92 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 183.11 KiB | 16.65 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [2]:
!pip install -q sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently ta

In [3]:
import sys
sys.path.insert(0, "/content/ragprojekt")

from src.parse_docling import load_corpus
els = load_corpus("/content/ragprojekt/data-parsed")
print(len(els), "elements")

601 elements


In [4]:
from transformers import AutoTokenizer
from src.chunking import chunk_fixed, chunk_recursive, chunk_structural

tok = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")

STRATEGIES = {
    "fixed": chunk_fixed,
    "recursive": chunk_recursive,
    "structural": chunk_structural,
}

chunk_sets = {name: fn(els, tok) for name, fn in STRATEGIES.items()}
for name, cs in chunk_sets.items():
    print(f"{name:11} {len(cs)} chunks")

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (585 > 512). Running this sequence through the model will result in indexing errors


fixed       143 chunks
recursive   83 chunks
structural  113 chunks


In [5]:
MODELS = {
    "e5-small": {
        "name": "intfloat/multilingual-e5-small",
        "passage_prefix": "passage: ",
        "query_prefix": "query: ",
    },
    "bge-m3": {
        "name": "BAAI/bge-m3",
        "passage_prefix": "",
        "query_prefix": "",
    },
    "minilm-en": {
        "name": "sentence-transformers/all-MiniLM-L6-v2",
        "passage_prefix": "",
        "query_prefix": "",
    },
}

In [6]:
from sentence_transformers import SentenceTransformer

for key, cfg in MODELS.items():
    m = SentenceTransformer(cfg["name"], device="cuda")
    print(f"{key:11} dim {m.get_embedding_dimension()}")
    del m

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

minilm-en   dim 384


In [9]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

cuda


In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
from src.evaluate import rank_of, score
from src.eval_specs import SPECS, QUESTIONS

K = 5

def run(chunks, cfg, model):
    passages = [cfg["passage_prefix"] + c["text"] for c in chunks]
    P = model.encode(passages, normalize_embeddings=True,
                     batch_size=32, show_progress_bar=False)

    results = []
    for qid, question in QUESTIONS.items():
        q = model.encode(cfg["query_prefix"] + question, normalize_embeddings=True)
        sims = P @ q
        top = np.argsort(-sims)[:K]
        retrieved = [chunks[i] for i in top]
        results.append((qid, rank_of(retrieved, SPECS[qid])))
    return results

In [10]:
rows = []
for mkey, cfg in MODELS.items():
    model = SentenceTransformer(cfg["name"], device=DEVICE)
    for skey, chunks in chunk_sets.items():
        res = run(chunks, cfg, model)
        s = score(res)
        rows.append({"model": mkey, "chunking": skey, **s,
                     "ranks": {q: r for q, r in res}})
        print(f"{mkey:11} {skey:11} hit {s['hit_rate']:.2f}  MRR {s['mrr']:.3f}")
    del model
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

e5-small    fixed       hit 0.55  MRR 0.455
e5-small    recursive   hit 1.00  MRR 0.597
e5-small    structural  hit 0.91  MRR 0.697


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

bge-m3      fixed       hit 0.64  MRR 0.455
bge-m3      recursive   hit 0.91  MRR 0.632
bge-m3      structural  hit 1.00  MRR 0.768


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

minilm-en   fixed       hit 0.55  MRR 0.253
minilm-en   recursive   hit 0.64  MRR 0.377
minilm-en   structural  hit 0.73  MRR 0.594


In [15]:
import json, pandas as pd
df = pd.DataFrame([{k: v for k, v in r.items() if k != "ranks"} for r in rows])
df.to_csv("/content/ragprojekt/notes/week2-comparison.csv", index=False)

ranks = pd.DataFrame([{"model": r["model"], "chunking": r["chunking"], **r["ranks"]}
                      for r in rows])
ranks.to_csv("/content/ragprojekt/notes/week2-ranks.csv", index=False)
print(df.to_string(index=False))
print()
print(ranks.to_string(index=False))

    model   chunking  n  hit_rate      mrr
 e5-small      fixed 11  0.545455 0.454545
 e5-small  recursive 11  1.000000 0.596970
 e5-small structural 11  0.909091 0.696970
   bge-m3      fixed 11  0.636364 0.454545
   bge-m3  recursive 11  0.909091 0.631818
   bge-m3 structural 11  1.000000 0.768182
minilm-en      fixed 11  0.545455 0.253030
minilm-en  recursive 11  0.636364 0.377273
minilm-en structural 11  0.727273 0.593939

    model   chunking  Q1  Q2  Q5  Q6  Q7  Q8  Q9  Q10  Q11  Q12  Q13
 e5-small      fixed NaN 1.0   2   2   1 1.0 NaN  NaN    1  NaN  NaN
 e5-small  recursive 5.0 3.0   1   1   1 5.0 4.0  1.0    1  4.0  3.0
 e5-small structural 3.0 2.0   1   1   1 1.0 NaN  1.0    1  2.0  3.0
   bge-m3      fixed 2.0 NaN   2   2   1 1.0 NaN  NaN    1  2.0  NaN
   bge-m3  recursive 2.0 5.0   1   2   1 NaN 2.0  1.0    1  4.0  1.0
   bge-m3 structural 1.0 5.0   1   1   1 1.0 4.0  2.0    1  2.0  1.0
minilm-en      fixed NaN 2.0   5   3   1 4.0 NaN  NaN    2  NaN  NaN
minilm-en  recurs

In [14]:
PAIRS = [
    ("skal vs bør, same clause",
     "Leverandøren skal sikre, at 15 % af de leverede årsværk udgøres af personer under oplæring",
     "Leverandøren bør sikre, at 15 % af de leverede årsværk udgøres af personer under oplæring"),
    ("unrelated clauses, same document",
    "Leverandøren skal sikre, at 15 % af de leverede årsværk udgøres af personer under oplæring",
    "Fakturering skal ske til AAU under overholdelse af lov om offentlige betalinger"),
]


for mkey, cfg in MODELS.items():
    m = SentenceTransformer(cfg["name"], device=DEVICE)
    for label, a, b in PAIRS:
        va, vb = m.encode([cfg["passage_prefix"] + a, cfg["passage_prefix"] + b],
                          normalize_embeddings=True)
        print(f"{mkey:11} {label}: {float(va @ vb):.4f}")
    del m
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

e5-small    skal vs bør, same clause: 0.9981
e5-small    unrelated clauses, same document: 0.8850


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

bge-m3      skal vs bør, same clause: 0.9920
bge-m3      unrelated clauses, same document: 0.4834


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

minilm-en   skal vs bør, same clause: 0.9685
minilm-en   unrelated clauses, same document: 0.5691


In [16]:
import pandas as pd

sim_rows = []
for mkey, cfg in MODELS.items():
    m = SentenceTransformer(cfg["name"], device=DEVICE)
    for label, a, b in PAIRS:
        va, vb = m.encode([cfg["passage_prefix"] + a, cfg["passage_prefix"] + b],
                          normalize_embeddings=True)
        sim_rows.append({"model": mkey, "pair": label, "similarity": round(float(va @ vb), 4)})
    del m
    torch.cuda.empty_cache()

sim = pd.DataFrame(sim_rows)
sim.to_csv("/content/ragprojekt/notes/week2-similarity.csv", index=False)
print(sim.to_string(index=False))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

    model                             pair  similarity
 e5-small         skal vs bør, same clause      0.9981
 e5-small unrelated clauses, same document      0.8850
   bge-m3         skal vs bør, same clause      0.9920
   bge-m3 unrelated clauses, same document      0.4834
minilm-en         skal vs bør, same clause      0.9685
minilm-en unrelated clauses, same document      0.5691


In [ ]:
#!git -C /content/ragprojekt log --oneline -2

b3492ae (HEAD -> main, origin/main, origin/HEAD) edited chunking to deal with page issue
0821c31 Week 2: retrieval scorer (hit rate, MRR) and machine-readable eval specs


In [ ]:
#print(len(chunk_sets["fixed"]))

123


In [ ]:
#!git log --oneline -2
#!git status --short

b3492ae (HEAD -> main, origin/main, origin/HEAD) edited chunking to deal with page issue
0821c31 Week 2: retrieval scorer (hit rate, MRR) and machine-readable eval specs


In [ ]:
#import inspect
#from src.chunking import chunk_fixed
#rint(inspect.getsource(chunk_fixed))

def chunk_fixed(elements, tok, size=250, overlap=50):
    """Week 1 baseline, in tokens rather than characters.

    Cuts every `size` tokens within a page, ignoring section, paragraph and
    sentence boundaries. Chunking per page (as Week 1 did) preserves page
    provenance; only the section metadata is unavailable.
    """
    chunks = []
    for source in sorted({e["source"] for e in elements}):
        els = [e for e in elements if e["source"] == source]
        pages = sorted({e["page"] for e in els if e["page"] is not None})
        for page in pages:
            page_els = [e for e in els if e["page"] == page]
            text = "\n".join(e["text"] for e in page_els)
            ids = tok.encode(text, add_special_tokens=False)
            step = size - overlap
            for i in range(0, len(ids), step):
                window = ids[i:i + size]
                if not window:
                    continue
                chunks.append({
                    "text": tok.decode(w

In [ ]:
import pandas as pd
df = pd.DataFrame([{"model": r["model"], "chunking": r["chunking"], **r["ranks"]}
                   for r in rows])
print(df[df.chunking == "structural"].to_string(index=False))

    model   chunking  Q1  Q2  Q5  Q6  Q7  Q8  Q9  Q10  Q11  Q12  Q13
 e5-small structural 3.0 2.0 1.0 1.0 1.0 1.0 NaN  1.0  1.0  2.0  3.0
   bge-m3 structural 1.0 5.0 1.0 1.0 1.0 1.0 4.0  2.0  1.0  2.0  1.0
minilm-en structural NaN 1.0 1.0 1.0 1.0 5.0 NaN  1.0  1.0  NaN  3.0


In [ ]:
#c = chunk_sets["fixed"][0]
#print({k: v for k, v in c.items() if k != "text"})

{'source': 'Aftale', 'page': 1, 'section': None, 'section_path': None, 'subsection': None, 'delaftale': None, 'element_type': 'fixed', 'strategy': 'fixed'}


In [ ]:
#from src.evaluate import matches
#from src.eval_specs import SPECS

#for qid, spec in SPECS.items():
#    if spec is None:
#        continue
#    n = sum(1 for c in chunk_sets["fixed"] if matches(c, spec))
#    print(f"{qid:4} {n} matching chunks")

Q1   1 matching chunks
Q2   1 matching chunks
Q5   0 matching chunks
Q6   0 matching chunks
Q7   0 matching chunks
Q8   0 matching chunks
Q9   0 matching chunks
Q10  0 matching chunks
Q11  0 matching chunks
Q12  0 matching chunks
Q13  0 matching chunks
